In [1]:
# =============================================================================
# STEP 7 - VALIDATING THE LABEL-FREE MONITOR
#
# The monitor is the study's only constructive contribution and is currently
# labelled exploratory because it was developed and evaluated on the same three
# environments, with no external check and no comparison against simpler signals.
# This notebook addresses four gaps:
#
#   (1) BASELINES. Is predicted-class score drift needed, or would mean-confidence
#       shift, entropy shift, predicted-share drift alone, or raw score KS do as
#       well? If a one-line baseline matches it, the monitor is not a contribution.
#   (2) ALL-NEGATIVE ENVIRONMENT. CIC-IoT-2023 has no failing class, which is
#       exactly where specificity and false-alarm burden are measured. Excluding
#       it inflates the monitor's apparent value.
#   (3) LEAVE-ONE-ENVIRONMENT-OUT. Fix the rule on two failing environments, test
#       on the third, rotate.
#   (4) OPERATING POINTS. AUROC is threshold-free; an operator needs a threshold,
#       a false-alarm rate and a recall.
#
# WHAT LOEO CAN AND CANNOT ESTABLISH. The monitor has no fitted parameters in the
# usual sense, but its DESIGN was chosen with all three environments in view: the
# predicted-class score signal, the mass-collapse fallback, and within-environment
# percentile ranking. LOEO tests the COMBINATION RULE on held-out data; it cannot
# undo the fact that the signal family was selected knowing these datasets. We
# state that rather than claim full out-of-sample validation.
#
# THIS CAN GO AGAINST THE MONITOR. If a baseline matches it, or if it fails to
# transfer, the notebook says so and the paper must follow.
# =============================================================================
import numpy as np, pandas as pd
from scipy import stats
from sklearn.metrics import roc_auc_score
import os, sys, json, shutil, glob, subprocess, hashlib, time
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT=Path('/content/drive/MyDrive'); PARENT_DIR=DRIVE_ROOT/'CALSHIFT_Research'
PROJECT_ROOT=PARENT_DIR/'calshift-research'; CRED_DIR=DRIVE_ROOT/'.gitcreds'
assert PROJECT_ROOT.exists(), 'Drive mount unhealthy; restart runtime and remount'
subprocess.run(['git','config','--global','user.name','Md Anas Biswas'],check=False)
subprocess.run(['git','config','--global','user.email','anasbiswas@gmail.com'],check=False)
subprocess.run(['git','config','--global','credential.helper','store'],check=False)
for fn,dest in [('.git-credentials','/root/.git-credentials'),('.gitconfig','/root/.gitconfig')]:
    for cand in (PARENT_DIR/fn, CRED_DIR/fn):
        if cand.exists(): shutil.copy(cand,dest); os.chmod(dest,0o600); break
os.chdir(PROJECT_ROOT); sys.path.insert(0,str(PROJECT_ROOT/'src'))
import importlib
for m in ['config','conformal']:
    if m in sys.modules: importlib.reload(sys.modules[m])
import config, conformal
RD=config.REPORTS_DIR; ALPHA=config.ALPHA_PRIMARY
MIN_SUPPORT=20; FAIL_THRESHOLD=0.05
print('ready | alpha', ALPHA, '| min predicted support', MIN_SUPPORT)


Mounted at /content/drive
ready | alpha 0.05 | min predicted support 20


In [2]:
# =============================================================================
# Cell 2 - signals. The monitor exactly as nb21 defines it, plus four baselines
# computed on the same restriction (target flows PREDICTED as class c versus
# source flows predicted as class c), so the comparison is like for like.
# =============================================================================
def aps_all(P):
    o=np.argsort(-P,axis=1); sp=np.take_along_axis(P,o,1); cum=np.cumsum(sp,1)
    ss=cum-0.5*sp; out=np.empty_like(P); np.put_along_axis(out,o,ss,1); return out

def ks(a,b):
    if len(a)<5 or len(b)<5: return np.nan
    allv=np.sort(np.concatenate([a,b]))
    ca=np.searchsorted(np.sort(a),allv,side='right')/len(a)
    cb=np.searchsorted(np.sort(b),allv,side='right')/len(b)
    return float(np.max(np.abs(ca-cb)))

def entropy(P):
    Q=np.clip(P,1e-12,None); return -(Q*np.log(Q)).sum(1)

def signal_rows(dataset, arch, classes, P_src, P_tgt, cov_lookup):
    """Monitor signal plus four baselines, per class, all label-free."""
    Ssrc, Stgt = aps_all(P_src), aps_all(P_tgt)
    yh_s, yh_t = P_src.argmax(1), P_tgt.argmax(1)
    conf_s, conf_t = P_src.max(1), P_tgt.max(1)
    ent_s, ent_t = entropy(P_src), entropy(P_tgt)
    out=[]
    for ci,cn in enumerate(classes):
        ms, mt = yh_s==ci, yh_t==ci
        n_pred_t=int(mt.sum())
        share_s, share_t = float(ms.mean()), float(mt.mean())
        low = n_pred_t < MIN_SUPPORT
        row={'dataset':dataset,'arch':arch,'class':cn,'n_pred_target':n_pred_t,
             'low_support':bool(low),
             # --- the monitor ---
             'monitor_drift': ks(Ssrc[ms,ci], Stgt[mt,ci]) if not low else np.nan,
             'pred_mass_drop': share_s-share_t,
             # --- baselines, same restriction, all label-free ---
             'b_confidence_shift': abs(float(conf_s[ms].mean())-float(conf_t[mt].mean())) if not low and ms.any() else np.nan,
             'b_entropy_shift':    abs(float(ent_s[ms].mean())-float(ent_t[mt].mean()))   if not low and ms.any() else np.nan,
             'b_share_drift':      abs(share_s-share_t),
             'b_raw_score_ks':     ks(Ssrc[:,ci], Stgt[:,ci]),
             'coverage': cov_lookup.get(cn, np.nan)}
        row['undercoverage']=(1-ALPHA)-row['coverage'] if row['coverage']==row['coverage'] else np.nan
        out.append(row)
    return out

def check_classes(npz, assumed, tag):
    """The saved column order is authoritative. Hardcoding it and hoping is how class
    labels get silently misassigned, so read it back and assert."""
    if 'classes' in npz.files:
        saved=[str(x) for x in npz['classes']]
        assert saved==list(assumed), f'{tag}: saved class order {saved} != assumed {list(assumed)}'
        return saved
    return list(assumed)

SIGNALS=['monitor','b_confidence_shift','b_entropy_shift','b_share_drift','b_raw_score_ks']
print('signals defined:', SIGNALS)
print('  monitor = within-environment rank of predicted-class score drift,')
print('            with mass-collapse fallback where predicted support is low')


signals defined: ['monitor', 'b_confidence_shift', 'b_entropy_shift', 'b_share_drift', 'b_raw_score_ks']
  monitor = within-environment rank of predicted-class score drift,
            with mass-collapse fallback where predicted support is low


In [ ]:
# =============================================================================
# Cell 3 - compute signals for all FOUR environments, including CIC-IoT-2023,
# which has never been run through the monitor and is the all-negative case.
# =============================================================================
def cov_map(f, rung=None):
    d=pd.read_csv(RD/f); d=d[(np.isclose(d.alpha,ALPHA))&(d.protocol=='SHC')]
    if 'feasible' in d.columns: d=d[d['feasible']]
    if rung is not None and 'rung' in d.columns: d=d[np.isclose(d['rung'],rung)]
    return d.groupby('class')['coverage'].mean().to_dict()

rows=[]; t0=time.time()

# ---------- NSL-KDD (rung 0.80) ----------
CL=config.CANONICAL_CLASSES; c2i={c:i for i,c in enumerate(CL)}
tr=pd.read_parquet(config.INTERIM_DIR/'nslkdd_train.parquet').reset_index(drop=True)
te=pd.read_parquet(config.INTERIM_DIR/'nslkdd_test.parquet').reset_index(drop=True)
part=pd.read_parquet(config.PROC_DIR/'nslkdd_source_partition_labels.parquet')
tr=tr.assign(partition=part['partition'].values)
assign=pd.read_parquet(config.PROC_DIR/'nslkdd_ladder_assignments.parquet')
IDX={(r,j,role):g['test_idx'].to_numpy() for (r,j,role),g in assign.groupby(['rung','realization','role'])}
REALS=sorted(assign[np.isclose(assign.rung,0.80)]['realization'].unique())
cm=cov_map('coverage_primary_nslkdd.csv', 0.80)
for f in sorted(glob.glob(str(config.PROC_DIR/'probs_*.npz'))):
    arch,seed=Path(f).stem.replace('probs_','').rsplit('_s',1)
    d=np.load(f); P_sp=d['S_pool'].astype(np.float64); P_te=d['target'].astype(np.float64)
    for j in REALS[:5]:
        ev=IDX.get((0.80,j,'eval'))
        if ev is None or len(ev)==0: continue
        rows += signal_rows('nslkdd', arch, CL, P_sp, P_te[ev], cm)
print(f'  nslkdd rows {sum(1 for r in rows if r["dataset"]=="nslkdd")}')

# ---------- UGR'16 ----------
UGR=config.DATASETS_DIR/'ugr16'
us=pd.read_parquet(UGR/'july_week5.parquet'); ut=pd.read_parquet(UGR/'august_week1.parquet')
for dd in (us,ut): dd['label']=dd['label'].astype(str).str.strip().str.lower()
UK=['background','dos','scan11','scan44','nerisbotnet']
us=us[us.label.isin(UK)].reset_index(drop=True); ut=ut[ut.label.isin(UK)].reset_index(drop=True)
UCL=sorted(UK)
cm=cov_map('coverage_primary_ugr16.csv')
for f in sorted((config.DATA_DIR/'ugr16_probs').glob('ugr16__*.npz')):
    _,arch,sd=Path(f).stem.split('__'); d=np.load(f)
    cls=check_classes(d, UCL, 'ugr16')
    rows += signal_rows('ugr16', arch, cls, d['srcpool'].astype(np.float64), d['target'].astype(np.float64), cm)
print(f'  ugr16 rows {sum(1 for r in rows if r["dataset"]=="ugr16")}')

# ---------- CIC-IDS2017 ----------
cm=cov_map('coverage_primary_cicids2017.csv')
CCL=['Benign','DoS']
REAL=['R1_holdout_Slowhttptest','R2_holdout_Slowloris','R3_holdout_GoldenEye',
      'R4_holdout_Slowloris_Slowhttptest','R5_holdout_GoldenEye_Slowloris']
for name in REAL:
    for f in sorted((config.DATA_DIR/'cic_probs').glob(f'{name}__*.npz')):
        _,arch,sd=Path(f).stem.split('__'); d=np.load(f)
        cls=check_classes(d, CCL, 'cicids2017')
        rows += signal_rows('cicids2017', arch, cls, d['srcpool'].astype(np.float64),
                            d['target'].astype(np.float64), cm)
print(f'  cicids2017 rows {sum(1 for r in rows if r["dataset"]=="cicids2017")}')

# ---------- CIC-IoT-2023: the all-negative environment ----------
ICL=json.loads((RD/'ciciot2023_model_record.json').read_text())['classes_canonical_order']
cm=cov_map('coverage_primary_ciciot2023.csv', 0.80)
for f in sorted((config.DATA_DIR/'ciciot_probs').glob('ciciot2023__*.npz')):
    _,arch,sd=Path(f).stem.split('__'); d=np.load(f)
    cls=check_classes(d, ICL, 'ciciot2023')
    rows += signal_rows('ciciot2023', arch, cls, d['srcpool'].astype(np.float64),
                        d['target'].astype(np.float64), cm)
print(f'  ciciot2023 rows {sum(1 for r in rows if r["dataset"]=="ciciot2023")}')
S=pd.DataFrame(rows)
assert len(S)>0, 'no signal rows produced; check the probability-file globs'
# classes with no coverage entry are those the feasibility rule excluded (NSL U2R).
# Report the drop rather than letting a name mismatch vanish silently.
dropped=S[S.coverage.isna()].groupby(['dataset','class']).size()
if len(dropped):
    print('\nclasses with no coverage entry, dropped:')
    print(dropped.to_string())
    print('  (expected: NSL-KDD U2R, excluded by the feasibility rule. Anything else here')
    print('   is a class-name mismatch and must be investigated before trusting the output.)')
S=S[S.coverage.notna()].reset_index(drop=True)
assert len(S)>0, 'every row dropped: coverage lookup failed, likely a class-name mismatch'
S['failing']=(S.undercoverage>FAIL_THRESHOLD).astype(int)
print(f'\ntotal {len(S)} class-model rows | {time.time()-t0:.0f}s')
print(S.groupby('dataset').agg(rows=('class','size'), failing=('failing','sum')).to_string())


  nslkdd rows 750
  ugr16 rows 150
  cicids2017 rows 300


In [ ]:
# =============================================================================
# Cell 4 - (1) BASELINES. Does the monitor beat simpler label-free signals?
# Aggregated to one row per dataset-class so the model panel is not treated as
# independent replication.
# =============================================================================
agg=S.groupby(['dataset','class'],as_index=False).agg(
    monitor_drift=('monitor_drift','mean'), pred_mass_drop=('pred_mass_drop','mean'),
    b_confidence_shift=('b_confidence_shift','mean'), b_entropy_shift=('b_entropy_shift','mean'),
    b_share_drift=('b_share_drift','mean'), b_raw_score_ks=('b_raw_score_ks','mean'),
    low_support=('low_support','max'), undercoverage=('undercoverage','mean'),
    failing=('failing','max'))

def build_detector(df, signal):
    """Within-environment percentile rank, with the mass-collapse fallback for the monitor.
    Requires a 0..n-1 index: out[g.index] indexes positionally, so a filtered frame with a
    gapped index would write to the wrong rows or raise."""
    assert list(df.index)==list(range(len(df))), 'build_detector needs a reset 0..n-1 index'
    out=np.full(len(df), np.nan)
    for ds,g in df.groupby('dataset'):
        if signal=='monitor':
            rd=g['monitor_drift'].rank(pct=True)
            rm=g['pred_mass_drop'].clip(lower=0).rank(pct=True)
            v=np.where(g['low_support'].fillna(False), rm, rd)
        else:
            v=g[signal].rank(pct=True).to_numpy()
        out[g.index]=v
    return out

print('BASELINE COMPARISON (pooled over all four environments)')
print(f"{'signal':22s} {'rho':>7s} {'AUROC':>7s} {'n':>4s}")
res=[]
for sig in SIGNALS:
    d=build_detector(agg, sig); ok=~np.isnan(d)
    if ok.sum()<6: continue
    y=agg.loc[ok,'failing']
    r,_=stats.spearmanr(d[ok], agg.loc[ok,'undercoverage'])
    a=roc_auc_score(y, d[ok]) if y.nunique()>1 else np.nan
    res.append({'signal':sig,'rho':r,'auroc':a,'n':int(ok.sum())})
    print(f"  {sig:20s} {r:+7.3f} {a:7.3f} {int(ok.sum()):4d}")
B=pd.DataFrame(res)
assert (B.signal=='monitor').any(), 'monitor row missing from the comparison'
mon=float(B[B.signal=='monitor'].auroc.iloc[0])
base_vals=B[B.signal!='monitor'].auroc.dropna()
assert len(base_vals)>0, 'no baseline produced a usable AUROC; comparison impossible'
assert mon==mon, 'monitor AUROC is NaN; comparison impossible'
best_base=float(base_vals.max())
best_name=B.loc[B[B.signal!='monitor'].auroc.idxmax(),'signal']
print(f"\n  monitor {mon:.3f} vs best baseline {best_base:.3f} ({best_name})  "
      f"margin {mon-best_base:+.3f}")
if mon <= best_base + 0.02:
    print('  A SIMPLER SIGNAL MATCHES THE MONITOR. The predicted-class score drift is not')
    print('  earning its complexity, and the paper should say so and report the simpler one.')
else:
    print('  The monitor beats the simplest label-free alternatives on this pooled comparison.')


In [ ]:
# =============================================================================
# Cell 5 - (2) ALL-NEGATIVE SPECIFICITY and (3) LEAVE-ONE-ENVIRONMENT-OUT.
# =============================================================================
FAILING_ENVS=[ds for ds,g in agg.groupby('dataset') if g.failing.sum()>0]
print('environments with failing classes:', FAILING_ENVS)
print('all-negative environments:', [d for d in agg.dataset.unique() if d not in FAILING_ENVS])

print('\n(2) SPECIFICITY ON THE ALL-NEGATIVE ENVIRONMENT')
agg['det_monitor']=build_detector(agg,'monitor')
neg=agg[(~agg.dataset.isin(FAILING_ENVS)) & agg.det_monitor.notna()]
if len(neg):
    # threshold chosen on the failing environments, applied to the all-negative one
    pos=agg[agg.dataset.isin(FAILING_ENVS) & agg.det_monitor.notna()]
    for q in [0.50,0.60,0.70,0.80]:
        thr=q
        fa=float((neg.det_monitor>=thr).mean())
        rec=float((pos[pos.failing==1].det_monitor>=thr).mean())
        print(f"  threshold {thr:.2f}: recall on failing envs {rec:.3f} | "
              f"false-alarm rate on the all-negative env {fa:.3f} "
              f"({int((neg.det_monitor>=thr).sum())} of {len(neg)} classes flagged)")
    print('\n  Every class in the all-negative environment is a true negative by construction,')
    print('  so any flag there is a false alarm. Excluding this environment, as the earlier')
    print('  analysis did, reports recall without its cost.')
else:
    print('  no all-negative environment available')

print('\n(3) LEAVE-ONE-ENVIRONMENT-OUT')
print('  the rule is rebuilt on the held-in environments and applied to the held-out one')
loeo=[]
for held in FAILING_ENVS:
    tr_=agg[agg.dataset!=held]; te_=agg[agg.dataset==held].copy()
    # the rule has no fitted parameters; what transfers is the THRESHOLD, chosen on held-in
    d_tr=build_detector(tr_.reset_index(drop=True),'monitor')
    ok=~np.isnan(d_tr); y_tr=tr_.reset_index(drop=True).loc[ok,'failing']
    # threshold maximising Youden's J on held-in
    best_thr, best_j = 0.5, -1
    for thr in np.arange(0.05,1.0,0.05):
        pred=(d_tr[ok]>=thr).astype(int)
        tp=((pred==1)&(y_tr==1)).sum(); fn=((pred==0)&(y_tr==1)).sum()
        fp=((pred==1)&(y_tr==0)).sum(); tn=((pred==0)&(y_tr==0)).sum()
        j=(tp/max(tp+fn,1))+(tn/max(tn+fp,1))-1
        if j>best_j: best_j, best_thr = j, thr
    te_['det']=build_detector(te_.reset_index(drop=True),'monitor')
    ok2=te_.det.notna()
    y=te_.loc[ok2,'failing']; d=te_.loc[ok2,'det']
    auc=roc_auc_score(y,d) if y.nunique()>1 else np.nan
    pred=(d>=best_thr).astype(int)
    tp=int(((pred==1)&(y==1)).sum()); fn=int(((pred==0)&(y==1)).sum())
    fp=int(((pred==1)&(y==0)).sum()); tn=int(((pred==0)&(y==0)).sum())
    loeo.append({'held_out':held,'n':int(ok2.sum()),'n_failing':int(y.sum()),
                 'threshold_from_heldin':round(best_thr,2),'auroc':auc,
                 'recall':tp/max(tp+fn,1),'specificity':tn/max(tn+fp,1),'tp':tp,'fp':fp,'fn':fn,'tn':tn})
    print(f"  held out {held:12s} n={int(ok2.sum()):2d} failing={int(y.sum()):2d} "
          f"thr={best_thr:.2f} AUROC={auc if auc==auc else float('nan'):.3f} "
          f"recall={tp/max(tp+fn,1):.3f} spec={tn/max(tn+fp,1):.3f}")
L=pd.DataFrame(loeo)
print(f"\n  mean held-out AUROC: {L.auroc.mean():.3f} | mean recall {L.recall.mean():.3f} | "
      f"mean specificity {L.specificity.mean():.3f}")
print('  NOTE: this tests the COMBINATION RULE on held-out data. The signal family and the')
print('  mass-collapse fallback were chosen with all environments in view, so this is not')
print('  full out-of-sample validation and is not reported as such.')


In [ ]:
# =============================================================================
# Cell 6 - (4) OPERATING POINTS, then save and commit.
# =============================================================================
print('OPERATING POINTS (all four environments pooled)')
d=agg.det_monitor; ok=d.notna()
y=agg.loc[ok,'failing']; dv=d[ok]
print(f"{'threshold':>10s} {'flagged':>8s} {'recall':>7s} {'precision':>10s} {'specificity':>12s} {'false alarms':>13s}")
ops=[]
for thr in [0.40,0.50,0.60,0.70,0.80,0.90]:
    pred=(dv>=thr).astype(int)
    tp=int(((pred==1)&(y==1)).sum()); fp=int(((pred==1)&(y==0)).sum())
    fn=int(((pred==0)&(y==1)).sum()); tn=int(((pred==0)&(y==0)).sum())
    rec=tp/max(tp+fn,1); prec=tp/max(tp+fp,1); spec=tn/max(tn+fp,1)
    ops.append({'threshold':thr,'flagged':int(pred.sum()),'recall':rec,'precision':prec,
                'specificity':spec,'false_alarms':fp,'tp':tp,'fn':fn,'tn':tn})
    print(f"{thr:10.2f} {int(pred.sum()):8d} {rec:7.3f} {prec:10.3f} {spec:12.3f} {fp:13d}")
O=pd.DataFrame(ops)
print('\n  read: at a given threshold an operator flags this many classes for review,')
print('        catches this fraction of genuinely failing ones, and raises this many false alarms')

S.to_csv(RD/'monitor_signals_all_environments.csv', index=False)
agg.to_csv(RD/'monitor_class_level.csv', index=False)
B.to_csv(RD/'monitor_baseline_comparison.csv', index=False)
L.to_csv(RD/'monitor_loeo.csv', index=False)
O.to_csv(RD/'monitor_operating_points.csv', index=False)
(RD/'monitor_validation_verdict.json').write_text(json.dumps({
 'scope':'four environments including CIC-IoT-2023 as an all-negative case',
 'failing_environments':FAILING_ENVS,
 'baselines':B.round(4).to_dict('records'),
 'monitor_vs_best_baseline':{'monitor':float(mon),'best_baseline':float(best_base),
                             'margin':float(mon-best_base)},
 'loeo':L.round(4).to_dict('records'),
 'loeo_mean_auroc':float(L.auroc.mean()),
 'operating_points':O.round(4).to_dict('records'),
 'caveat':'LOEO tests the combination rule on held-out data. The signal family and the '
          'mass-collapse fallback were selected with all environments in view, so this is '
          'not full out-of-sample validation.'}, indent=2, default=str))
print('\nsaved five artefacts and the verdict')

def git(*a, show=True):
    r=subprocess.run(['git',*a],capture_output=True,text=True)
    if show and (r.stdout or r.stderr): print((r.stdout+r.stderr).strip())
    return r
for s,dd in [('/root/.git-credentials',PARENT_DIR/'.git-credentials'),('/root/.gitconfig',PARENT_DIR/'.gitconfig')]:
    if os.path.exists(s): shutil.copy(s,dd)
os.chdir(PROJECT_ROOT)
lock=PROJECT_ROOT/'.git'/'index.lock'
if lock.exists() and not subprocess.run(['pgrep','git'],capture_output=True).stdout.strip():
    lock.unlink(); print('removed stale git lock')
for attempt in (1,2):
    git('add','-A',show=False)
    if git('status','--porcelain',show=False).stdout.strip():
        git('commit','-m','step 7: monitor validation - baselines, all-negative specificity, leave-one-environment-out, operating points')
        r=git('push','-u','origin','main')
        if r.returncode: print('PUSH FAILED. Commit is safe locally.')
        break
    if attempt==1: print('waiting 10s for Drive sync...'); time.sleep(10)
    else: print('nothing to commit')
print(git('log','--oneline','-3',show=False).stdout)
